# Rebuild `psu_dsacsma_hf` from PSU's raw output

Legacy's `psu_dsacsma_hf` (and so the copy made by `09_psu_simulations.ipynb`) holds the
**dPRMS** values: the TEEHR-formatted file it was loaded from,
`teehr_format/psu_dsacsma_hf.parquet`, is a copy of `psu_dprms_hf.parquet`. PSU's raw
`raw_data/sacsma_sim.zip` is right. This notebook rebuilds the dataset from it and replaces
`public.psu_dsacsma_hf`'s rows.

The raw files (`<model>_sim.zip`, one `Qs.csv` each) hold one row per gage, in
`gages2700.csv`'s order, and one column per day from 1992-10-01, in mm/day (the README says
mm/s; the factor shows mm/day). Streamflow is

    value [m³/s] = Qs [mm/day] × DRAIN_SQKM × 1000 / 86400

**3. Check the formula** rebuilds dPRMS from its raw file and compares it, gage by gage, with
`public.psu_dprms_hf` (right in legacy): it writes nothing. **4. Replace** overwrites every month
of `psu_dsacsma_hf` (the same gages and days; the series table doesn't change).

**Run it as a platform admin, on use1.** Re-runnable.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs
from teehr.evaluation.spark_session_utils import create_spark_session

## 1. Settings

In [ ]:
#: PSU's raw output (us-east-2; its objects aren't public). If reading them fails, copy
#: gages2700.csv, prms_sim.zip and sacsma_sim.zip to a directory here and set RAW to it.
RAW = ("s3://ciroh-rti-public-data/teehr-data-warehouse/common/simulations/"
       "psu_differentiable_simulations/raw_data")
FIRST_DAY = "1992-10-01"
PUBLIC = "iceberg.public"
REGION = "us-east-2"
DATASET, CHECK = "psu_dsacsma_hf", "psu_dprms_hf"
WORK = tempfile.mkdtemp()

In [ ]:
spark = create_spark_session()  # local mode: 17.8 M rows
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")

def raw(name):
    """A local path to one of RAW's files (downloaded to WORK when RAW is in S3)."""
    if not RAW.startswith("s3://"):
        return os.path.join(RAW, name)
    local = os.path.join(WORK, name)
    if not os.path.exists(local):
        fs.copy_files(f"{RAW.removeprefix('s3://')}/{name}", local,
                      source_filesystem=fs.S3FileSystem(region=REGION),
                      destination_filesystem=fs.LocalFileSystem())
    return local


gages = pd.read_csv(raw("gages2700.csv"), dtype={"STAID": str})
KEYS = "usgs-" + gages.STAID.str.zfill(8)
FACTOR = (gages.DRAIN_SQKM * 1000 / 86400).to_numpy()[:, None]
print(f"{len(gages):,} gages")


def streamflow(model):
    """A model's raw Qs.csv in m³/s: one row per gage (KEYS), one column per day."""
    qs = pd.read_csv(raw(f"{model}_sim.zip"), header=None).to_numpy()
    assert qs.shape[0] == len(KEYS), qs.shape
    assert not np.isnan(qs).any()
    return qs * FACTOR


def days(flow):
    """The days of a model's columns (UTC midnights, as naive timestamps)."""
    return pd.date_range(FIRST_DAY, periods=flow.shape[1], freq="D")

## 2. Read

In [ ]:
%%time
prms, sacsma = streamflow("prms"), streamflow("sacsma")
print(f"{prms.shape[1]:,} days, {days(prms)[0]:%Y-%m-%d} to {days(prms)[-1]:%Y-%m-%d}")
print(f"dSAC-SMA differs from dPRMS in {(sacsma != prms).mean():.1%} of values")

## 3. Check the formula against `psu_dprms_hf` (writes nothing)

In [ ]:
%%time
v1 = spark.sql(f"""
    SELECT series_key, count(*) AS n, sum(value) AS total, min(value_time) AS first,
           max(value_time) AS last
    FROM {PUBLIC}.{CHECK} GROUP BY 1
""").toPandas().set_index("series_key").loc[KEYS]
rel = (v1.total.to_numpy() - prms.sum(axis=1)) / np.maximum(prms.sum(axis=1), 1e-9)
print(f"rows per gage match: {(v1.n == prms.shape[1]).all()}; "
      f"period matches: {(v1['first'] == days(prms)[0]).all() and (v1['last'] == days(prms)[-1]).all()}")
print(f"largest relative difference of a gage's total: {np.abs(rel).max():.1e} (float32 in legacy)")
assert np.abs(rel).max() < 1e-5

## 4. Replace `psu_dsacsma_hf`

Every month is in the new rows, so `overwritePartitions` replaces the whole table's data in one
commit.

In [ ]:
%%time
path = os.path.join(WORK, f"{DATASET}.parquet")
pq.write_table(pa.table({
    "series_key": pa.array(np.repeat(KEYS.to_numpy(), sacsma.shape[1])),
    "value_time": pa.array(np.tile(days(sacsma).to_numpy(), sacsma.shape[0]),
                           type=pa.timestamp("us", tz="UTC")),
    "value": pa.array(sacsma.ravel()),
}), path)
spark.read.parquet(path).writeTo(f"{PUBLIC}.{DATASET}").overwritePartitions()
print(f"{DATASET}: replaced with {sacsma.size:,} rows")

## 5. Check

In [ ]:
%%time
spark.sql(f"""
    SELECT 'dsacsma' AS dataset, count(*) AS rows, count(DISTINCT series_key) AS gages,
           min(value_time) AS first, max(value_time) AS last, sum(value) AS total
    FROM {PUBLIC}.{DATASET}
    UNION ALL
    SELECT 'dprms', count(*), count(DISTINCT series_key), min(value_time), max(value_time),
           sum(value)
    FROM {PUBLIC}.{CHECK}
""").show(truncate=False)
print(f"expected dsacsma total: {sacsma.sum():,.1f}")

In [ ]:
spark.stop()